In [6]:
import os
import sys
from pathlib import Path


# **Primo test Baseline**


In [7]:
#(LOCAL vs COLAB)
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("🌍 Detetected enviroment: COLAB")
    from google.colab import drive
    drive.mount('/content/drive')
    input_dir = Path('')
    output_dir = Path('')
else:
    print("💻 Detetected enviroment: LOCAL")
    # Impostate la vostra cartella di input e output locale
    input_dir = Path('../data/')
    output_dir = Path('../data/output')

    # Crea le cartelle se non esistono in locale
    output_dir.mkdir(parents=True, exist_ok=True)
    
INPUT_DIR, OUTPUT_DIR = input_dir, output_dir
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


💻 Detetected enviroment: LOCAL
Input directory: ..\data
Output directory: ..\data\output


## Setup API milionario


In [8]:
#OCCHIO CHE VB AGGIUNGE SYS PATH QUINDI METTETE PERCORSO GIUSTO (teoricamente è giusto già)

# Define the path to the directory containing your package
package_parent_dir = './'
package_parent_dir = os.path.abspath(package_parent_dir)
#print(package_parent_dir)

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)
    

# Verify the path was added
#print(sys.path)


#import your package
from millionaire_client import MillionaireClient, AuthenticationError
import dotenv
dotenv.load_dotenv()

#credentiali per accedere al server
username = os.getenv('USER')
pwd = os.getenv('PASSWORD')
mail=os.getenv('MAIL')
API_URL = "http://131.175.15.22:51111/"

# Create a client instance
client = MillionaireClient(API_URL)
print(f"username: {username}, password: {pwd}, mail: {mail}, API_URL: {API_URL}")
try:
    # Authenticate with the server
    user = client.login(username, pwd)
    print("✅ Login successful :)")
    print(f"\n Welcome, {user.username}! Role: {user.role}")
except AuthenticationError as e:
    print(f"❌ Login failed: {e}")
    
    
    
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")




username: Alisss, password: alicebella!, mail: alice.berta@mail.polimi.it, API_URL: http://131.175.15.22:51111/
✅ Login successful :)

 Welcome, Alisss! Role: student

=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [9]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

### **PLAY**


In [10]:
# Choose a competition ID
comp_id = 1

# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===
Session ID: 6088
Total number of questions: 15


--- Level 1 ---
Q: What was Nero's primary method for gaining popularity among the lower-class citizens of Rome?

  [0] Promoting public entertainments
  [1] Building new temples
  [2] Increasing military spending
  [3] Reducing taxes for the wealthy

Time remaining: 30.2s
TIMED OUT!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


In [11]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. AleAssini: $1,024,000.00 (Level 15)
  2. supreme_leader: $1,024,000.00 (Level 15)
  3. luca_bordin: $1,024,000.00 (Level 15)
  4. Anonymous: $1,024,000.00 (Level 15)
  5. Jasmin: $1,024,000.00 (Level 15)
